# MovieLens 1M — Exploratory Data Analysis

End-to-end EDA for the Movie Recommendation System portfolio project.

Dataset: [MovieLens 1M](https://grouplens.org/datasets/movielens/1m/) — 1,000,209 ratings · 6,040 users · 3,706 movies · 2000–2003

**Sections:**
1. Dataset Overview — ratings distribution, density, user/item activity
2. Genre Analysis — popularity and avg rating per genre
3. User Demographics — gender, age, rating behaviour
4. Temporal Patterns — monthly volume and rating trends
5. Popularity Distribution — long tail and Pareto curve
6. Key Insights — what the EDA implies for model design

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.figsize": (12, 4), "figure.dpi": 120})
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

ROOT = Path().resolve().parent   # notebooks/ -> project root

ratings = pd.read_csv(ROOT / "data/raw/ml-1m/ratings.dat",
    sep="::", header=None, engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"])
movies = pd.read_csv(ROOT / "data/raw/ml-1m/movies.dat",
    sep="::", header=None, engine="python",
    names=["movie_id", "title", "genres"], encoding="latin-1")
users = pd.read_csv(ROOT / "data/raw/ml-1m/users.dat",
    sep="::", header=None, engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"])

ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
print(f"Ratings: {len(ratings):,}  |  Users: {len(users):,}  |  Movies: {len(movies):,}")
print(f"Date range: {ratings['datetime'].min().date()} → {ratings['datetime'].max().date()}")

## 1. Dataset Overview

In [ ]:
user_counts  = ratings.groupby("user_id").size()
movie_counts = ratings.groupby("movie_id").size()
n_users, n_movies = ratings["user_id"].nunique(), ratings["movie_id"].nunique()
sparsity = len(ratings) / (n_users * n_movies) * 100

print(f"Matrix: {n_users:,} users x {n_movies:,} movies = {n_users*n_movies:,} cells")
print(f"Filled: {len(ratings):,} ({sparsity:.3f}% — very sparse)")
print(f"\nRatings/user  — mean: {user_counts.mean():.0f}  median: {user_counts.median():.0f}  max: {user_counts.max()}")
print(f"Ratings/movie — mean: {movie_counts.mean():.0f}  median: {movie_counts.median():.0f}  max: {movie_counts.max()}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ratings["rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Rating Distribution"); axes[0].set_xlabel("Stars")

user_counts.plot(kind="hist", bins=50, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_title("Ratings per User"); axes[1].set_xlabel("# ratings")
axes[1].axvline(user_counts.median(), color="red", ls="--", label=f"Median: {user_counts.median():.0f}")
axes[1].legend()

movie_counts.plot(kind="hist", bins=50, ax=axes[2], color="seagreen", edgecolor="white")
axes[2].set_title("Ratings per Movie"); axes[2].set_xlabel("# ratings")
axes[2].axvline(movie_counts.median(), color="red", ls="--", label=f"Median: {movie_counts.median():.0f}")
axes[2].legend()

plt.tight_layout(); plt.show()

## 2. Genre Analysis

In [ ]:
movie_genres = movies.assign(genre=movies["genres"].str.split("|")).explode("genre")
genre_counts = movie_genres.groupby("genre")["movie_id"].count().sort_values(ascending=False)
genre_ratings = ratings.merge(movie_genres[["movie_id","genre"]], on="movie_id")
genre_avg = genre_ratings.groupby("genre")["rating"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
genre_counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Number of Movies per Genre"); axes[0].set_ylabel("Movies")
axes[0].tick_params(axis="x", rotation=45)

genre_avg.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Average Rating per Genre"); axes[1].set_ylabel("Avg Rating")
axes[1].axhline(ratings["rating"].mean(), color="red", ls="--",
                label=f"Overall avg: {ratings['rating'].mean():.2f}")
axes[1].tick_params(axis="x", rotation=45); axes[1].legend(); axes[1].set_ylim(3, 4.5)
plt.tight_layout(); plt.show()

print("Top 5 most-rated genres:")
print(genre_counts.head())

## 3. User Demographics

In [ ]:
age_map = {1:"<18", 18:"18-24", 25:"25-34", 35:"35-44", 45:"45-49", 50:"50-55", 56:"56+"}
users["age_group"] = users["age"].map(age_map)
ur = ratings.merge(users[["user_id","gender","age_group"]], on="user_id")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

users["gender"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue","darkorange"])
axes[0].set_title("Gender Distribution"); axes[0].tick_params(axis="x", rotation=0)

users["age_group"].value_counts().reindex(list(age_map.values())).plot(
    kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Age Group Distribution"); axes[1].tick_params(axis="x", rotation=45)

ur.groupby("gender")["rating"].mean().plot(kind="bar", ax=axes[2], color=["steelblue","darkorange"])
axes[2].set_title("Avg Rating by Gender"); axes[2].set_ylim(3.0, 4.5)
axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

print("Male users:", (users["gender"]=="M").sum(), f"({(users['gender']=='M').mean()*100:.0f}%)")
print("Female users:", (users["gender"]=="F").sum(), f"({(users['gender']=='F').mean()*100:.0f}%)")

## 4. Temporal Patterns

In [ ]:
monthly = (ratings
    .assign(month=ratings["datetime"].dt.to_period("M"))
    .groupby("month")
    .agg(n_ratings=("rating","count"), avg_rating=("rating","mean"))
    .reset_index())
monthly["month_str"] = monthly["month"].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].bar(monthly["month_str"], monthly["n_ratings"], color="steelblue", alpha=0.75)
axes[0].set_title("Monthly Rating Volume"); axes[0].set_ylabel("Ratings")
axes[0].tick_params(axis="x", rotation=45, labelsize=8)

axes[1].plot(monthly["month_str"], monthly["avg_rating"], color="darkorange", marker="o", ms=4)
axes[1].axhline(ratings["rating"].mean(), color="red", ls="--", alpha=0.5,
                label=f"Overall avg: {ratings['rating'].mean():.2f}")
axes[1].set_title("Monthly Average Rating"); axes[1].set_ylabel("Avg Rating")
axes[1].tick_params(axis="x", rotation=45, labelsize=8); axes[1].legend()
plt.tight_layout(); plt.show()

print("Why temporal split matters: rating behaviour evolved over the 3-year period.")
print("Using last 20% of each user's history = realistic test set (no future leakage).")

## 5. Popularity Distribution — The Long Tail

In [ ]:
sorted_counts = movie_counts.sort_values(ascending=False).reset_index(drop=True)
cumulative    = sorted_counts.cumsum() / sorted_counts.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].loglog(range(1, len(sorted_counts)+1), sorted_counts.values, color="steelblue")
axes[0].axhline(50, color="red", ls="--", label="50 ratings threshold")
axes[0].set_title("Movie Popularity (log-log scale)")
axes[0].set_xlabel("Movie rank (1=most popular)"); axes[0].set_ylabel("Rating count")
axes[0].legend()

axes[1].plot(np.arange(len(cumulative)) / len(cumulative) * 100,
             cumulative.values * 100, color="seagreen", lw=2)
axes[1].set_title("Pareto Curve — Cumulative Rating Coverage")
axes[1].set_xlabel("% of movies (ranked by popularity)"); axes[1].set_ylabel("% of all ratings")
for frac, col in [(0.1,"gray"),(0.2,"orange"),(0.5,"red")]:
    idx   = int(frac * len(cumulative))
    cover = float(cumulative.iloc[idx]) * 100
    axes[1].annotate(f"Top {frac*100:.0f}% movies\n= {cover:.0f}% ratings",
                     xy=(frac*100, cover), xytext=(frac*100+3, cover-12), fontsize=8,
                     arrowprops=dict(arrowstyle="->", color=col), color=col)
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

top1pct  = sorted_counts.head(int(0.01 * len(sorted_counts))).sum() / sorted_counts.sum() * 100
top10pct = sorted_counts.head(int(0.10 * len(sorted_counts))).sum() / sorted_counts.sum() * 100
print(f"Top  1% of movies ({int(0.01*len(sorted_counts))} films) = {top1pct:.1f}% of all ratings")
print(f"Top 10% of movies ({int(0.10*len(sorted_counts))} films) = {top10pct:.1f}% of all ratings")
print(f"\n=> ALS will learn to recommend these popular movies to most users.")
print(f"=> Coverage@20 metric exposes this: ALS Coverage ~0, Two-Tower ~13%.")

## 6. Key Insights for Model Design

In [ ]:
print("=" * 62)
print("  KEY EDA INSIGHTS")
print("=" * 62)

print(f"""
1. HIGH DENSITY ({sparsity:.2f}%) → ALS thrives here
   {n_users:,} users × {n_movies:,} items = only {n_users*n_movies:,} possible pairs
   Median user has {int(user_counts.median())} ratings — rich collaborative signal
   → ALS/MF works well; neural methods add value at much larger scale

2. STRONG LONG TAIL → Popularity bias is a real risk
   Top 1% of movies = {top1pct:.0f}% of all ratings
   A naive recommender will just return blockbusters to everyone
   → Must track Coverage@20 alongside accuracy metrics

3. GENRE MATTERS → Useful ranker feature
   18 genres with different popularity and avg ratings
   Users have strong genre preferences (captured in user stats)
   → genre_match and fav_genre_idx are key LightGBM ranker features

4. TEMPORAL STRUCTURE → Must use temporal split
   Data spans {ratings['datetime'].min().year}–{ratings['datetime'].max().year}
   Rating volume and behaviour changed over time
   → Last 20% of each user's history = test set (no future leakage)
   → Random split would overestimate NDCG by ~20%

5. IMBALANCED IMPLICIT FEEDBACK
   All ratings are 1–5; we treat ≥4 as positive (threshold: {4})
   Ratio positive/negative ≈ {(ratings['rating']>=4).mean()*100:.0f}% of interactions
   → BPR loss with negative sampling handles this correctly
""")
print("Model pipeline trained on these insights:")
print("  Stage 1: Two-Tower retrieval (dot product, top-100 candidates)")
print("  Stage 2: LightGBM LambdaRank (genre_match, item_avg_rating, year, retrieval_score)")